# Prithvi EO 2.0 — Fine-Tuning for Segmentation

This notebook fine-tunes Prithvi EO 2.0 for land-cover segmentation using:
- **peft LoRA**: Adapts only a small number of parameters (~1-2% of total), dramatically reducing VRAM and training time
- **FCN decoder**: A simple fully-convolutional decoder head attached to the ViT encoder

LoRA (Low-Rank Adaptation) freezes the pretrained backbone and adds small trainable rank-decomposition matrices to attention layers. This is the recommended approach when labeled data is limited (<1000 samples).

## References
- peft LoRA: https://huggingface.co/docs/peft/conceptual_guides/lora
- Prithvi fine-tuning guide: https://github.com/NASA-IMPACT/hls-foundation-os

## 1. Setup

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from transformers import AutoModel
from peft import LoraConfig, get_peft_model, TaskType

sys.path.insert(0, os.path.abspath("../../.."))
try:
    from dotenv_config import HF_TOKEN
except ImportError:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Load Prithvi and Apply LoRA

In [ ]:
model_id = "ibm-nasa-geospatial/Prithvi-EO-2.0-300M"

backbone = AutoModel.from_pretrained(
    model_id, token=HF_TOKEN or None, trust_remote_code=True
)

# Apply LoRA: adapt only the attention query and value projections
lora_config = LoraConfig(
    r=16,                          # LoRA rank
    lora_alpha=32,                 # scaling factor
    target_modules=["query", "value"],  # which layers to adapt
    lora_dropout=0.1,
    bias="none",
)

backbone = get_peft_model(backbone, lora_config)
backbone.print_trainable_parameters()

## 3. Define a Segmentation Model

In [ ]:
class PrithviSegmentation(nn.Module):
    def __init__(self, backbone, num_classes, embed_dim=768, img_size=224):
        super().__init__()
        self.backbone = backbone
        self.num_classes = num_classes
        self.img_size = img_size

        # Simple FCN decoder
        self.decoder = nn.Sequential(
            nn.ConvTranspose2d(embed_dim, 256, kernel_size=4, stride=4),
            nn.ReLU(),
            nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2),
            nn.ReLU(),
            nn.Conv2d(128, num_classes, kernel_size=1),
            nn.Upsample(size=(img_size, img_size), mode="bilinear", align_corners=False),
        )

    def forward(self, pixel_values):
        out = self.backbone(pixel_values=pixel_values)
        features = out.last_hidden_state  # [B, N_patches, embed_dim]

        # Reshape patch tokens to spatial grid
        n_patches = features.shape[1]
        grid_size = int(n_patches ** 0.5)
        features = features.permute(0, 2, 1).reshape(
            features.shape[0], -1, grid_size, grid_size
        )

        return self.decoder(features)  # [B, num_classes, H, W]


num_classes = 5  # e.g., water, forest, cropland, urban, bare soil
seg_model = PrithviSegmentation(backbone, num_classes=num_classes).to(device)

trainable = sum(p.numel() for p in seg_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in seg_model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({trainable/total*100:.1f}%)")

## 4. Training Loop

Using simulated data for demonstration. In practice, replace with real HLS chips and pixel-level labels.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

# Simulate a small labeled dataset (50 chips)
n_samples, num_frames, n_bands, img_size = 50, 3, 6, 224
X = torch.randn(n_samples, num_frames, n_bands, img_size, img_size)
y = torch.randint(0, num_classes, (n_samples, img_size, img_size))

dataset = TensorDataset(X, y)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

optimizer = torch.optim.AdamW(seg_model.parameters(), lr=1e-4)
criterion = nn.CrossEntropyLoss()

n_epochs = 3  # use more epochs for real training (20–50+)
losses = []

seg_model.train()
for epoch in range(n_epochs):
    epoch_loss = 0.0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = seg_model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
    avg_loss = epoch_loss / len(loader)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{n_epochs}  loss={avg_loss:.4f}")

plt.plot(losses)
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Prithvi LoRA fine-tuning loss")
plt.tight_layout()
plt.show()

## 5. Save the LoRA Adapter

Only the LoRA weights (a few MB) need to be saved — the frozen backbone can be reloaded from HuggingFace.

In [ ]:
os.makedirs("prithvi_lora_adapter", exist_ok=True)
seg_model.backbone.save_pretrained("prithvi_lora_adapter")
torch.save(seg_model.decoder.state_dict(), "prithvi_lora_adapter/decoder.pth")

adapter_files = os.listdir("prithvi_lora_adapter")
total_size = sum(
    os.path.getsize(f"prithvi_lora_adapter/{f}")
    for f in adapter_files
) / 1e6

print(f"Saved LoRA adapter ({total_size:.1f} MB total):")
for f in adapter_files:
    print(f"  {f}")
print("\nTo reload: AutoModel.from_pretrained(model_id) + PeftModel.from_pretrained(backbone, 'prithvi_lora_adapter')")

## 6. Real-World Fine-Tuning Tips

| Setting | Recommendation |
|---|---|
| LoRA rank | 8–32 (higher = more capacity, more params) |
| Learning rate | 1e-4 for LoRA, 1e-5 for decoder |
| Batch size | 4–8 on a 16 GB GPU |
| Epochs | 50–100 for small datasets (<500 chips) |
| Augmentation | Random flip, rotate, color jitter on bands |
| Validation | Hold out 20% for early stopping |

For the complete Prithvi fine-tuning pipeline, see the official NASA-IMPACT repository:
https://github.com/NASA-IMPACT/hls-foundation-os